# Practical 3 — Training and Fine-Tuning

**Large Language Models in Finance · Chapter 3 / Lecture 3**

Three of the chapter's central cost-and-training results, built in code: the parameter
arithmetic of **LoRA** (why low-rank adapters make fine-tuning cheap), **Chinchilla**
compute budgeting (how to split a FLOP budget between model size and data), and a tiny
**gradient-descent fine-tuning loop** that you watch converge on a toy regression.

**Setup.** Everything below is fully offline and `numpy`-only — no model downloads, no
network, no GPU. Each part ends in an `assert`-based self-check; run the cells top to
bottom and every part should print `... OK`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
print('numpy', np.__version__, '— offline practical, no other dependencies')

## Part 1 — Counting LoRA parameters  `[B]`

Full fine-tuning updates every weight in a layer's $d\times d$ matrix. LoRA freezes that
matrix and instead learns a low-rank update $\Delta W = BA$ with $A\in\mathbb{R}^{r\times d}$
and $B\in\mathbb{R}^{d\times r}$, so each adapted layer trains only $2dr$ parameters
instead of $d^2$.

**Your task:** complete `count_params(d, r, layers)` to return a tuple
`(full, lora, ratio)` — the trainable parameter count under full fine-tuning, under LoRA,
and the reduction ratio `full / lora`.

In [ ]:
def count_params(d, r, layers):
    # TODO: full updates d*d per layer; LoRA updates 2*d*r per layer
    full = layers * d * d
    lora = layers * 2 * d * r
    ratio = full / lora
    return full, lora, ratio

# A GPT-style stack: 96 adapted projection matrices of width d=12288, rank r=8.
d, r, layers = 12288, 8, 96
full, lora, ratio = count_params(d, r, layers)
print(f'full fine-tuning : {full:,} trainable params')
print(f'LoRA (r={r})      : {lora:,} trainable params')
print(f'reduction ratio  : {ratio:,.0f}x fewer trainable params')

In [ ]:
# self-check (Part 1)
assert count_params(d, r, layers) == (layers*d*d, layers*2*d*r, (d*d)/(2*d*r))
# LoRA must be strictly cheaper whenever r < d/2, and the ratio is exactly d/(2r)
f, l, ro = count_params(d, r, layers)
assert l < f and np.isclose(ro, d / (2*r))
# rank r = d/2 is the break-even point: LoRA stops saving parameters
assert np.isclose(count_params(d, d//2, layers)[2], 1.0)
print('Part 1 OK')

## Part 2 — Chinchilla compute budgeting  `[I]`

Training a dense transformer costs about $C \approx 6ND$ FLOPs for $N$ parameters and $D$
tokens. Hoffmann et al. (2022) found the loss-minimising split holds $N$ and $D$ in a
roughly fixed ratio, so the compute-optimal point puts a constant fraction of the budget
into each. Take the symmetric Chinchilla rule $N^\* \propto C^{1/2}$, $D^\* \propto C^{1/2}$
with $D^\* \approx 20\,N^\*$ (about 20 tokens per parameter).

**Your task:** complete `optimal_split(C)` to return `(N, D)` satisfying both
$6ND = C$ and $D = 20N$.

In [ ]:
def optimal_split(C, tokens_per_param=20.0):
    # TODO: solve 6*N*D = C with D = tokens_per_param * N
    #   => 6 * N * (k*N) = C  =>  N = sqrt(C / (6*k))
    k = tokens_per_param
    N = np.sqrt(C / (6.0 * k))
    D = k * N
    return N, D

for C in [1e21, 1e23, 1e25]:
    N, D = optimal_split(C)
    print(f'C={C:.0e} FLOPs  ->  N={N/1e9:8.3f}B params, '
          f'D={D/1e9:9.2f}B tokens, D/N={D/N:.1f}')

In [ ]:
# self-check (Part 2)
for C in [1e21, 1e23, 1e25]:
    N, D = optimal_split(C)
    assert np.isclose(6 * N * D, C, rtol=1e-9)      # budget is exactly spent
    assert np.isclose(D / N, 20.0)                  # 20 tokens per parameter
# a 100x larger budget should grow N and D by 10x each (the sqrt scaling)
N1, D1 = optimal_split(1e21)
N2, D2 = optimal_split(1e23)
assert np.isclose(N2 / N1, 10.0) and np.isclose(D2 / D1, 10.0)
print('Part 2 OK')

## Part 3 — A fine-tuning loop, by hand  `[A]`

Fine-tuning is gradient descent on a loss surface. Strip away the transformer and the
mechanics are visible on a toy linear regression: predict $y = Xw^\*$ from noisy data by
minimising the mean squared error $\mathcal{L}(w)=\tfrac{1}{n}\lVert Xw-y\rVert^2$, whose
gradient is $\nabla\mathcal{L}(w)=\tfrac{2}{n}X^\top(Xw-y)$.

**Your task:** complete `mse` and the gradient step in `fit` so the loss decreases each
epoch.

In [ ]:
# toy data: a known target weight vector plus a little noise
n, p = 200, 4
X = rng.normal(size=(n, p))
w_star = np.array([1.5, -2.0, 0.5, 3.0])
y = X @ w_star + 0.01 * rng.normal(size=n)

def mse(w):
    # TODO: mean squared error of predictions X @ w against y
    r = X @ w - y
    return float(r @ r / n)

def fit(lr=0.05, epochs=200):
    w = np.zeros(p)
    losses = []
    for _ in range(epochs):
        losses.append(mse(w))
        # TODO: gradient of the MSE and one descent step
        grad = (2.0 / n) * X.T @ (X @ w - y)
        w = w - lr * grad
    return w, np.array(losses)

w_hat, losses = fit()
print(f'initial loss : {losses[0]:.4f}')
print(f'final loss   : {losses[-1]:.6f}')
print(f'recovered w  : {np.round(w_hat, 3)}  (target {w_star})')

In [ ]:
# self-check (Part 3)
assert losses[-1] < losses[0]                       # training reduced the loss
assert np.all(np.diff(losses) <= 1e-9)              # monotone non-increasing
assert np.allclose(w_hat, w_star, atol=1e-1)       # converged near the true weights
print('Part 3 OK')

## What you built

- LoRA parameter arithmetic and its $d/(2r)$ reduction ratio (`Part 1`)
- The Chinchilla compute-optimal split from the $C\approx 6ND$ rule (`Part 2`)
- A from-scratch gradient-descent fine-tuning loop you watched converge (`Part 3`)